# Clausius + CS Hybrid Scan

This notebook runs a clean `QMM` hybrid model with:

- Clausius attractive mean field
- Carnahan-Starling excluded volume
- symmetric ground-state fit
- liquid critical point
- pure hadronic EOS table
- asymmetric `a_n, a_{pn}, b_n, b_{pn}` fit
- fixed-`y` asymmetric quarkyonic profiles

The hybrid model name inside `QMM` is `clausius_cs`.


## Hybrid Model

The model used here is:

\[
U(n) = -
rac{n}{1 + c n}
\]

for the attractive mean field, together with the Carnahan-Starling repulsive mapping

\[
f_{\mathrm{CS}}(x) = \exp\left[-
rac{3x}{4-x} - 
rac{4x}{(4-x)^2}
ight], \qquad x = bn.
\]

So this is not the original pure Clausius model and not the original pure CS model. It is a new hybrid inside `QMM` called `clausius_cs`.


In [1]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def locate_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'src' / 'qmm').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate the QMM project root from this notebook.')


ROOT = locate_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qmm.config import load_run_config
from qmm.runner import run_workflow

BASE_CONFIG_PATH = ROOT / 'examples' / 'clausius_cs_hybrid_base.json'
OUTPUT_ROOT = ROOT / 'results' / 'generated' / 'clausius_cs_hybrid_scan'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    base_config = json.load(handle)

print('ROOT =', ROOT)
print('Base config =', BASE_CONFIG_PATH)
print(json.dumps(base_config, indent=2))


ROOT = /Users/dajuarez4/Documents/QuarkMatt/QMM
Base config = /Users/dajuarez4/Documents/QuarkMatt/QMM/examples/clausius_cs_hybrid_base.json
{
  "run_name": "clausius_cs_hybrid",
  "model": {
    "name": "clausius_cs",
    "parameter_search": {
      "enabled": true,
      "target_k0": 280.0,
      "parameter_min": 0.0,
      "parameter_max": 6.0,
      "scan_steps": 121,
      "tol": 1e-08
    }
  },
  "workflows": {
    "ground_state": true,
    "quantum_critical": true,
    "hadronic_eos_table": true,
    "symmetric_quarkyonic": false,
    "asymmetric_fit": true,
    "asymmetric_quarkyonic": true,
    "neutron_star": false
  },
  "physical": {
    "n0": 0.16,
    "binding_energy": -16.0,
    "symmetry_energy_j": 32.5,
    "symmetry_slope_l": 58.9
  },
  "hadronic_eos": {
    "n_min_ratio": 0.2,
    "n_max_ratio": 5.0,
    "n_points": 160,
    "smoothing_window": 9,
    "smoothing_degree": 3,
    "derivative_floor": 1e-10
  },
  "quarkyonic": {
    "n_min_ratio": 1e-06,
    "n_max_ra

## Scan Inputs

The default scan uses the requested endpoint incompressibilities `K0 = [240, 320] MeV`.
Replace the list below if you want a denser scan.


In [12]:
# Example denser scan:
target_k0_values = [240.0, 260.0, 280.0, 300.0, 320.0]
# target_k0_values = [240.0, 320.0]

proton_fraction_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda_momentum_mev = 300.0
use_quick_quantum = True
write_plots = True

print('K0 targets:', target_k0_values)
print('fixed-y values:', proton_fraction_values)
print('lambda [MeV]:', lambda_momentum_mev)
print('quantum.quick =', use_quick_quantum)


K0 targets: [240.0, 260.0, 280.0, 300.0, 320.0]
fixed-y values: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda [MeV]: 300.0
quantum.quick = True


## Build One Config Per Target

This helper starts from the executable base JSON and only overrides the run-specific pieces.


In [13]:
def build_config_for_target_k0(target_k0: float) -> dict:
    config = deepcopy(base_config)
    run_slug = f'clausius_cs_k0_{int(round(target_k0))}'

    config['run_name'] = run_slug
    config['model']['name'] = 'clausius_cs'
    config['model']['parameter_search']['enabled'] = True
    config['model']['parameter_search']['target_k0'] = float(target_k0)
    config['model']['parameter_search']['parameter_min'] = 0.0
    config['model']['parameter_search']['parameter_max'] = 6.0
    config['model']['parameter_search']['scan_steps'] = 121

    config['workflows']['ground_state'] = True
    config['workflows']['quantum_critical'] = True
    config['workflows']['hadronic_eos_table'] = True
    config['workflows']['asymmetric_fit'] = True
    config['workflows']['asymmetric_quarkyonic'] = True
    config['workflows']['symmetric_quarkyonic'] = False
    config['workflows']['neutron_star'] = False

    config.setdefault('quantum', {})
    config['quantum']['quick'] = bool(use_quick_quantum)

    config['quarkyonic']['lambda_momentum_mev'] = float(lambda_momentum_mev)
    config['asymmetric']['enabled'] = True
    config['asymmetric']['branch_mode'] = 'target_l'
    config['asymmetric']['target_j'] = 32.5
    config['asymmetric']['target_l'] = 58.9
    config['asymmetric']['target_k0'] = float(target_k0)
    config['asymmetric']['proton_fraction_values'] = list(proton_fraction_values)
    config['asymmetric']['beta_equilibrium'] = False

    run_dir = OUTPUT_ROOT / f'K0_{int(round(target_k0))}'
    config['output']['directory'] = str(run_dir)
    config['output']['write_json'] = True
    config['output']['write_csv'] = True
    config['output']['write_plots'] = bool(write_plots)

    return config


## Run The Scan

This cell writes one JSON file per target `K0`, runs the workflow, and stores the returned summaries.


In [14]:
scan_summaries = {}
failures = []
config_dir = OUTPUT_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

for target_k0 in target_k0_values:
    config_dict = build_config_for_target_k0(target_k0)
    config_path = config_dir / f'clausius_cs_k0_{int(round(target_k0))}.json'
    config_path.write_text(json.dumps(config_dict, indent=2), encoding='utf-8')

    try:
        run_config = load_run_config(config_path)
        summary = run_workflow(run_config)
        scan_summaries[target_k0] = summary
        print(f'finished K0={target_k0:.1f} MeV -> c={summary["ground_state"]["parameter_value"]:.6f}')
    except Exception as exc:
        failures.append((target_k0, str(exc)))
        print(f'failed K0={target_k0:.1f} MeV -> {exc}')

if failures:
    print('failures detected:')
    for item in failures:
        print(item)
else:
    print('all runs completed')


finished K0=240.0 MeV -> c=4.725737
finished K0=260.0 MeV -> c=4.228923
finished K0=280.0 MeV -> c=3.784976
finished K0=300.0 MeV -> c=3.385153
finished K0=320.0 MeV -> c=3.022605
all runs completed


## Collect Summaries

The next cell extracts the main outputs: the resolved Clausius parameter `c`, the symmetric `a,b`, the liquid critical point, and the asymmetric couplings.


In [15]:
ground_rows = []
quantum_rows = []
asym_rows = []

for target_k0, summary in sorted(scan_summaries.items()):
    ground = dict(summary['ground_state'])
    ground['target_k0_requested'] = target_k0
    ground_rows.append(ground)

    quantum = dict(summary['quantum_critical'])
    quantum['target_k0_requested'] = target_k0
    quantum_rows.append(quantum)

    asym = dict(summary['asymmetric_fit'])
    asym['target_k0_requested'] = target_k0
    asym_rows.append(asym)

ground_df = pd.DataFrame(ground_rows).sort_values('target_k0_requested').reset_index(drop=True)
quantum_df = pd.DataFrame(quantum_rows).sort_values('target_k0_requested').reset_index(drop=True)
asym_df = pd.DataFrame(asym_rows).sort_values('target_k0_requested').reset_index(drop=True)

ground_df.to_csv(OUTPUT_ROOT / 'ground_state_summary.csv', index=False)
quantum_df.to_csv(OUTPUT_ROOT / 'quantum_critical_summary.csv', index=False)
asym_df.to_csv(OUTPUT_ROOT / 'asymmetric_fit_summary.csv', index=False)

display(ground_df[['target_k0_requested', 'parameter_value', 'a', 'b', 'K0', 'n_sat', 'binding']])
display(quantum_df[['target_k0_requested', 'Tc', 'nc', 'Pc', 'iterations', 'score']])
display(asym_df[['target_k0_requested', 'a_n', 'a_pn', 'b_n', 'b_pn', 'ratio_a_pn_over_a_n', 'ratio_b_pn_over_b_n', 'J', 'L']])


,target_k0_requested,parameter_value,a,b,K0,n_sat,binding
0,240.0,4.725737,475.386029,1.928645,239.999998,0.16,-16.0
1,260.0,4.228923,461.868049,2.141070,259.999998,0.16,-16.0
2,280.0,3.784976,449.772374,2.338330,279.999999,0.16,-16.0
3,300.0,3.385153,438.868975,2.522526,300.000001,0.16,-16.0
4,320.0,3.022605,428.976869,2.695356,320.000001,0.16,-16.0


,target_k0_requested,Tc,nc,Pc,iterations,score
0,240.0,16.023033,0.049502,0.237733,11,2.031313e-13
1,260.0,16.235103,0.051100,0.252048,10,4.416504e-13
2,280.0,16.433595,0.052611,0.266113,12,8.300011e-14
3,300.0,16.620204,0.054044,0.279954,12,8.356855e-14
4,320.0,16.796345,0.055408,0.293594,12,1.326996e-13


,target_k0_requested,a_n,a_pn,b_n,b_pn,ratio_a_pn_over_a_n,ratio_b_pn_over_b_n,J,L
0,240.0,261.959579,688.812479,1.429716,2.427573,2.629461,1.697941,32.5,58.9
1,260.0,255.024483,668.711616,1.525135,2.757005,2.622147,1.807712,32.5,58.9
2,280.0,248.928057,650.616692,1.618420,3.058239,2.613674,1.889644,32.5,58.9
3,300.0,243.531401,634.206549,1.709786,3.335265,2.604209,1.950691,32.5,58.9
4,320.0,238.725775,619.227963,1.799412,3.591300,2.593888,1.995819,32.5,58.9


## Ground-State And Critical-Point Plots


In [16]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(ground_df['target_k0_requested'], ground_df['parameter_value'], marker='o', lw=2)
axes[0, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 0].set_ylabel('resolved c')
axes[0, 0].set_title('Clausius parameter from target K0')

axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['a'], marker='o', lw=2, label='a')
axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['b'], marker='s', lw=2, label='b')
axes[0, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 1].set_title('Symmetric nuclear-matter couplings')
axes[0, 1].legend(loc='best')

axes[1, 0].scatter(quantum_df['nc'], quantum_df['Tc'], c=quantum_df['target_k0_requested'], cmap='viridis', s=120)
for row in quantum_df.itertuples():
    axes[1, 0].annotate(f"K0={row.target_k0_requested:.0f}", (row.nc, row.Tc), xytext=(5, 5), textcoords='offset points', fontsize=8)
axes[1, 0].set_xlabel(r'$n_c$ [fm$^{-3}$]')
axes[1, 0].set_ylabel(r'$T_c$ [MeV]')
axes[1, 0].set_title(r'$T_c$ vs $n_c$')

axes[1, 1].plot(quantum_df['target_k0_requested'], quantum_df['Tc'], marker='o', lw=2, label=r'$T_c$')
axes[1, 1].plot(quantum_df['target_k0_requested'], quantum_df['nc'], marker='s', lw=2, label=r'$n_c$')
axes[1, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 1].set_title('Critical-point evolution')
axes[1, 1].legend(loc='best')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_75749/637996815.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Pure Hadronic EOS Table

Each run writes a dense zero-temperature hadronic EOS table before quarkyonic matter is introduced. The CSV includes the reconstructed hadronic sound speed.


In [17]:
selected_target_k0_hadronic = target_k0_values[0]
hadronic_summary = scan_summaries[selected_target_k0_hadronic]
hadronic_csv = Path(hadronic_summary['outputs']['hadronic_eos_csv'])
hadronic_df = pd.read_csv(hadronic_csv)

print('Hadronic EOS CSV:', hadronic_csv)
display(hadronic_df.head())


Hadronic EOS CSV: /Users/dajuarez4/Documents/QuarkMatt/QMM/results/generated/clausius_cs_hybrid_scan/K0_240/clausius_cs_k0_240_hadronic_eos.csv


,model,parameter_name,parameter_value,a,b,K0,n,n_over_n0,eps_raw,eps,e_per_particle_minus_m,mu_b,pressure,dP_dn,d2eps_dn2,vs2
0,clausius_cs,c,4.725737,475.386029,1.928645,239.999998,0.032000,0.200000,29.844453,29.844453,-5.360845,926.702168,-0.189984,-10.108592,-315.893504,-0.010908
1,clausius_cs,c,4.725737,475.386029,1.928645,239.999998,0.036830,0.230189,34.317022,34.317017,-6.236589,925.242794,-0.240150,-10.610904,-288.103431,-0.011468
2,clausius_cs,c,4.725737,475.386029,1.928645,239.999998,0.041660,0.260377,38.782854,38.782856,-7.070978,923.917886,-0.292088,-10.878950,-261.134205,-0.011775
3,clausius_cs,c,4.725737,475.386029,1.928645,239.999998,0.046491,0.290566,43.242599,43.242615,-7.862931,922.722714,-0.344714,-10.925973,-235.014837,-0.011841
4,clausius_cs,c,4.725737,475.386029,1.928645,239.999998,0.051321,0.320755,47.696882,47.696915,-8.612227,921.652384,-0.397019,-10.764662,-209.752614,-0.011680


In [18]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].plot(hadronic_df['n_over_n0'], hadronic_df['e_per_particle_minus_m'], lw=2.2)
axes[0].axhline(0.0, color='black', ls='--', lw=1.0)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$E/A - m_N$ [MeV]')
axes[0].set_title(f'Hadronic binding curve, K0={selected_target_k0_hadronic:.0f} MeV')

axes[1].plot(hadronic_df['n_over_n0'], hadronic_df['pressure'], lw=2.2)
axes[1].axhline(0.0, color='black', ls='--', lw=1.0)
axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1].set_title('Pure hadronic pressure')

axes[2].plot(hadronic_df['n_over_n0'], hadronic_df['vs2'], lw=2.2)
axes[2].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[2].set_ylim(-0.1, 0.8)
axes[2].set_xlabel(r'$n/n_0$')
axes[2].set_ylabel(r'$v_s^2$')
axes[2].set_title('Pure hadronic sound speed')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_75749/1287798810.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Asymmetric Couplings

This is where the notebook summarizes `a_n`, `a_{pn}`, `b_n`, `b_{pn}`, and the split ratios.


In [19]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

axes[0].plot(asym_df['target_k0_requested'], asym_df['a_n'], marker='o', lw=2, label=r'$a_n$')
axes[0].plot(asym_df['target_k0_requested'], asym_df['a_pn'], marker='s', lw=2, label=r'$a_{pn}$')
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'[MeV fm$^3$]')
axes[0].set_title('Attractive asymmetric couplings')
axes[0].legend(loc='best')

axes[1].plot(asym_df['target_k0_requested'], asym_df['b_n'], marker='o', lw=2, label=r'$b_n$')
axes[1].plot(asym_df['target_k0_requested'], asym_df['b_pn'], marker='s', lw=2, label=r'$b_{pn}$')
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'[fm$^3$]')
axes[1].set_title('Repulsive asymmetric couplings')
axes[1].legend(loc='best')

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(asym_df['target_k0_requested'], asym_df['ratio_a_pn_over_a_n'], marker='o', lw=2)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$a_{pn}/a_n$')
axes[0].set_title('Attractive split ratio')

axes[1].plot(asym_df['target_k0_requested'], asym_df['ratio_b_pn_over_b_n'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$b_{pn}/b_n$')
axes[1].set_title('Repulsive split ratio')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_75749/1137333372.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_75749/1137333372.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Fixed-`y` Asymmetric Quarkyonic Profiles

Pick one target `K0` below and the notebook will load all fixed-`y` outputs from that run.


In [20]:
selected_target_k0 = target_k0_values[0]
selected_summary = scan_summaries[selected_target_k0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for y_value in proton_fraction_values:
    key = f'asymmetric_y_{y_value:0.3f}_csv'
    csv_path = Path(selected_summary['outputs'][key])
    df = pd.read_csv(csv_path)
    label = f'y={y_value:.1f}'
    axes[0].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)
    axes[1].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)

axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$f_Q$')
axes[0].set_title(f'Quark fraction for K0={selected_target_k0:.0f} MeV')
axes[0].legend(loc='upper left', ncol=2)

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$v_s^2$')
axes[1].set_ylim(-0.1, 0.8)
axes[1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[1].set_title(f'Sound speed for K0={selected_target_k0:.0f} MeV')
axes[1].legend(loc='upper right', ncol=2)

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_75749/1031276873.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Output Check

This final cell lists the generated JSON inputs and output folders for the scan.


In [21]:
print('Generated config files:')
for path in sorted((OUTPUT_ROOT / 'configs').glob('*.json')):
    print(' -', path.relative_to(ROOT))

print('Output folders:')
for path in sorted(OUTPUT_ROOT.glob('K0_*')):
    if path.is_dir():
        print(' -', path.relative_to(ROOT))


Generated config files:
 - results/generated/clausius_cs_hybrid_scan/configs/clausius_cs_k0_240.json
 - results/generated/clausius_cs_hybrid_scan/configs/clausius_cs_k0_260.json
 - results/generated/clausius_cs_hybrid_scan/configs/clausius_cs_k0_280.json
 - results/generated/clausius_cs_hybrid_scan/configs/clausius_cs_k0_300.json
 - results/generated/clausius_cs_hybrid_scan/configs/clausius_cs_k0_320.json
Output folders:
 - results/generated/clausius_cs_hybrid_scan/K0_240
 - results/generated/clausius_cs_hybrid_scan/K0_260
 - results/generated/clausius_cs_hybrid_scan/K0_280
 - results/generated/clausius_cs_hybrid_scan/K0_300
 - results/generated/clausius_cs_hybrid_scan/K0_320
